In [3]:
%load_ext autoreload
%autoreload 2
%cd /opt/tiger/samantha

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
/opt/tiger/samantha


In [4]:
import json
import pandas as pd
from tqdm import tqdm
from samantha.utils.hdfs_helper import hdfs_ls

# MusicSFT (EN)

In [114]:
# index_files = hdfs_ls("hdfs://harunava/home/byte_data_seed_us/hdd_va/speech/data_store/BigMusic/music_sft-en-20231221_Smcc_Mvocal_N6.2k/index_1/*.parquet")
index_files = hdfs_ls("hdfs://harunava/home/byte_data_seed_us/hdd_va/speech/data_store/BigMusic/music_Smcc_Tplaylist_N30k/index_2/*.parquet")

df = pd.concat([pd.read_parquet(f) for f in tqdm(index_files)])
df = pd.io.json.json_normalize(df["meta"].apply(json.loads))


2024-01-17 03:39:55,083 - samantha.utils.hdfs_helper - INFO - Listing HDFS directory hdfs://harunava/home/byte_data_seed_us/hdd_va/speech/data_store/BigMusic/music_Smcc_Tplaylist_N30k/index_2/*.parquet


100%|██████████| 699/699 [00:22<00:00, 31.13it/s]
/tmp/ipykernel_9082/1013872161.py:5: FutureWarning: pandas.io.json.json_normalize is deprecated, use pandas.json_normalize instead.
  df = pd.io.json.json_normalize(df["meta"].apply(json.loads))


In [122]:
df[["playlist_extra.playlist_url", "playlist_extra.label1", "playlist_extra.label2"]].value_counts().reset_index().to_csv("mcc_30k_index_value_counts.csv", index=None)

In [35]:
# genre tags

## except: 中文场景 -> use label2 for SCENE
## except: 中文心情 -> use label2 for MOOD
## discard genre labels in label2

df["playlist_extra.label1"].value_counts()

中文场景    20011
中文心情     4837
流行       1549
嘻哈        791
下沉土嗨      751
国风音乐      532
摇滚        419
古典        400
爵士        319
中国传统      283
金属        183
儿童音乐      169
宗教         98
Name: playlist_extra.label1, dtype: int64

In [95]:

def change_label(row):
    if row["playlist_extra.label1"] == "中文场景":
        row["final_scene"] = row["playlist_extra.label2"]
        
    elif row["playlist_extra.label1"] == "中文心情":
        row["final_mood"] = row["playlist_extra.label2"]
    else:
        row["final_genre"] = row["playlist_extra.label1"]
        
    # pop -> more detailed
    if row["final_genre"] == "流行":
        row["final_genre"] = row["playlist_extra.label2"]
    return row

df = df.apply(change_label, axis=1)


In [172]:
from pprint import pprint

tokenizer = MusicSFTTokenizerGenresZH()
pprint(tokenizer.translation)


translation = []
for w in df["final_genre"].value_counts().keys().tolist():
    translation.append(tokenizer.translate(w))
    
ax = df["final_genre"].value_counts().plot.barh()
ax.set_yticks(range(len(translation)), translation)
ax.set_title("MCC Playlist 30k final_genre distribution")
plt.show()

print("Total tracks with final_genre label:", len(df.dropna(subset=["final_genre"])))
print("Total tracks with final_mood label:", len(df.dropna(subset=["final_mood"])))
print("Total tracks with final_scene label:", len(df.dropna(subset=["final_scene"])))


# ax = df["final_mood"].value_counts().plot.barh()
# ax.set_title("MCC Playlist 30k final_mood distribution")
# plt.show()

ax = df["final_scene"].value_counts().plot.barh(figsize=(10,10))
ax.set_title("MCC Playlist 30k final_scene distribution")
plt.show()

/opt/tiger/samantha/recipes/datasets/mir/music_sft.py:35: UserWarning: torchaudio._backend.set_audio_backend has been deprecated. With dispatcher enabled, this function is no-op. You can remove the function call.
  torchaudio.set_audio_backend("soundfile")


{'DJ慢摇/土味remix': 'dj',
 '下沉土嗨': 'tuhai',
 '中国传统': 'traditional',
 '中国戏曲': 'chinese opera',
 '中国风流行音乐': 'chinese pop',
 '传统民歌': 'traditional folk',
 '儿童音乐': 'children',
 '古典': 'classical',
 '古风音乐': 'ancient music',
 '喊麦': 'hanmai',
 '嘻哈': 'hippop',
 '国语流行': 'chinese pop',
 '国风嘻哈': 'chinese style hip-hop',
 '国风电子': 'guofeng electronics',
 '国风音乐': 'chinese pop',
 '宗教': 'religion',
 '律动': 'rhythm',
 '怀旧流行': 'nostalgic pop',
 '摇滚': 'rock',
 '放克音乐': 'funk',
 '旋律性说唱': 'melodic rap',
 '流行': 'pop',
 '流行民谣': 'folk',
 '流行爵士': 'jazz',
 '爵士': 'jazz',
 '粤语流行': 'cantopop',
 '老派说唱': 'old school rap',
 '节奏蓝调/灵魂': 'blues',
 '越南鼓': 'vietnamese drum',
 '金属': 'metal',
 '闽南语流行': 'minnan pop',
 '陷阱说唱': 'trap'}


KeyError: 'final_genre'

In [178]:
# genre mapping

from recipes.datasets.mir.taxonomies.music_sft_zh import MusicSFTTokenizerGenresAllZH

tokenizer = MusicSFTTokenizerGenresAllZH()


filtered_df = df[(df["playlist_extra.label1"] != "中文场景") & (df["playlist_extra.label1"] != "中文心情")]

filtered_df["playlist_extra.label1_en"] = filtered_df["playlist_extra.label1"].apply(tokenizer.translate)
filtered_df["playlist_extra.label2_en"] = filtered_df["playlist_extra.label2"].apply(lambda r: tokenizer.translate(r) if r is not None else r)

# for idx, row in filtered_df[["playlist_extra.playlist_url", "playlist_extra.label1", "playlist_extra.label2"]].value_counts().reset_index().iterrows():
#     print(tokenizer.translate(row["playlist_extra.label1"]), tokenizer.translate(row["playlist_extra.label2"]), row[0])
    
    
filtered_df[["playlist_extra.label1_en", "playlist_extra.label2_en"]].value_counts().reset_index().to_csv("mcc_30k_index_genre_values.csv", index=None)

/tmp/ipykernel_9082/3953310229.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df["playlist_extra.label1_en"] = filtered_df["playlist_extra.label1"].apply(tokenizer.translate)
/tmp/ipykernel_9082/3953310229.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df["playlist_extra.label2_en"] = filtered_df["playlist_extra.label2"].apply(lambda r: tokenizer.translate(r) if r is not None else r)


In [182]:
filtered_df[["playlist_extra.label1_en"]].value_counts()

playlist_extra.label1_en
Pop                         1549
Hip Hop                      791
Tuhai                        751
Chinese Style                532
Rock                         419
Classical                    400
Jazz                         319
Chinese Tradition            283
Metal                        183
Childhood                    169
Devotional                    98
dtype: int64

In [175]:
filtered_df[["playlist_extra.label1_en", "playlist_extra.label2_en"]].value_counts()

playlist_extra.label1_en  playlist_extra.label2_en
pop                       folk                        648
                          minnan pop                  547
tuhai                     dj                          477
hippop                    trap                        413
jazz                      jazz                        319
hippop                    melodic rap                 286
classical                 funk                        280
chinese pop               chinese pop                 174
pop                       cantopop                    168
                          nostalgic pop               161
tuhai                     hanmai                      158
traditional               traditional folk            154
chinese pop               ancient music               136
                          chinese style hip-hop       135
traditional               chinese opera               129
classical                 blues                       120
tuhai                

In [171]:
# mood mapping
from recipes.datasets.mir.taxonomies.music_sft_zh import MusicSFTTokenizerMoodsZH

tokenizer = MusicSFTTokenizerMoodsZH()

filtered_df = df[df["playlist_extra.label1"] == "中文心情"]
filtered_df["playlist_extra.label2_en"] = filtered_df["playlist_extra.label2"].apply(lambda r: tokenizer.translate(r) if r is not None else r)

# for idx, row in filtered_df[["playlist_extra.playlist_url", "playlist_extra.label1", "playlist_extra.label2"]].value_counts().reset_index().iterrows():
#     print(tokenizer.translate(row["playlist_extra.label1"]), tokenizer.translate(row["playlist_extra.label2"]), row[0])
    
    
filtered_df[["playlist_extra.label1", "playlist_extra.label2_en"]].value_counts().reset_index().to_csv("mcc_30k_index_mood_values.csv", index=None)

/tmp/ipykernel_9082/3236476009.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df["playlist_extra.label2_en"] = filtered_df["playlist_extra.label2"].apply(lambda r: tokenizer.translate(r) if r is not None else r)


In [166]:
# scene mapping
filtered_df = df[(df["playlist_extra.label1"] == "中文场景")]
filtered_df["playlist_extra.label2"].value_counts().reset_index().to_csv("mcc_30k_index_scene_values.csv", index=None)

In [5]:
from recipes.datasets.mir.music_sft import MCC30KDataModuleZH, MusicSFTTokenizerGenresZH, MusicSFTMCC30KPreprocessedDataModule

tokenizer = MusicSFTTokenizerGenresZH()

# datamodule = MCC30KDataModuleZH(
datamodule = MusicSFTMCC30KPreprocessedDataModule(
    tokenizer=tokenizer,
    sample_rate=24000,
    duration=30,
    batch_size=64,
    shuffle_buffer_size=0,
    resampled=False,
    shardshuffle=False,
    num_workers=0
)
train_loader = datamodule.train_dataloader()
batch = next(iter(train_loader))


2024-01-17 08:12:03,168 - databus.databus_cache - INFO - databus python cache flush thread begin


/opt/tiger/samantha/samantha/dataio/parquet/writer.py:17: UserWarning: simplejson not installed, potential NaN issue in IndexShardWriter.
  warnings.warn("simplejson not installed, potential NaN issue in IndexShardWriter.")


No module named 'flash_attn'


/opt/tiger/samantha/recipes/datasets/mir/music_sft.py:35: UserWarning: torchaudio._backend.set_audio_backend has been deprecated. With dispatcher enabled, this function is no-op. You can remove the function call.
  torchaudio.set_audio_backend("soundfile")


initializing train datasets...
adding dataset: /mnt/bn/janne-research-xl/data/music_sft/chinese/genre_1/Childhood/url2index.txt
adding dataset: /mnt/bn/janne-research-xl/data/music_sft/chinese/genre_1/Chinese Style, China-Wave/url2index.txt
adding dataset: /mnt/bn/janne-research-xl/data/music_sft/chinese/genre_1/Chinese Style, Chinoiserie Electronic/url2index.txt
adding dataset: /mnt/bn/janne-research-xl/data/music_sft/chinese/genre_1/Chinese Style, Chinoiserie Rap/url2index.txt
adding dataset: /mnt/bn/janne-research-xl/data/music_sft/chinese/genre_1/Chinese Style, GuFeng Music/url2index.txt
adding dataset: /mnt/bn/janne-research-xl/data/music_sft/chinese/genre_1/Chinese Tradition, Chinese Opera/url2index.txt
adding dataset: /mnt/bn/janne-research-xl/data/music_sft/chinese/genre_1/Chinese Tradition, Traditional Chinese Folk/url2index.txt
adding dataset: /mnt/bn/janne-research-xl/data/music_sft/chinese/genre_1/Classical, Funk/url2index.txt
adding dataset: /mnt/bn/janne-research-xl/data/

In [16]:
import IPython.display as ipd
for idx in range(len(batch.audio)):
    # ipd.display(ipd.Audio(batch.audio[idx], rate=24000))
    print(batch.tag_names[idx])

Tuhai, VinaHouse
Pop, Chinese Pop
Hip Hop, R&B Rap
Tuhai, MC
Pop, Cantopop
Pop, Pop Folk
Chinese Style, GuFeng Music
Classical, R&B/Soul
Jazz, Jazz Pop
Chinese Tradition, Traditional Chinese Folk
Hip Hop, Old School
Chinese Style, GuFeng Music
Classical, R&B/Soul
Metal
Hip Hop, R&B Rap
Hip Hop, Old School
Devotional
Chinese Style, GuFeng Music
Pop, Cantopop
Pop, Pop Folk
Chinese Tradition, Chinese Opera
Pop, Chinese Pop
Chinese Style, Chinoiserie Electronic
Hip Hop, Trap Rap
Chinese Tradition, Chinese Opera
Jazz, Jazz Pop
Metal
Pop, Chinese Pop
Pop, Contemporary Pop
Chinese Tradition, Chinese Opera
Pop, Taiwanese Pop
Pop, Cantopop
Chinese Style, GuFeng Music
Chinese Style, Chinoiserie Rap
Hip Hop, R&B Rap
Tuhai, MC
Jazz, Jazz Pop
Hip Hop, R&B Rap
Pop, Taiwanese Pop
Classical, R&B/Soul
Chinese Tradition, Traditional Chinese Folk
Childhood
Pop, Contemporary Pop
Classical, R&B/Soul
Hip Hop, Trap Rap
Hip Hop, R&B Rap
Rock
Childhood
Tuhai, VinaHouse
Tuhai, DJ
Pop, Pop Folk
Chinese Tradition

In [191]:
import matplotlib.pyplot as plt

print("Total duration", df["duration"].sum() / 3600, "hours")
# print("Unique tracks", df["song_id"].unique().shape[0])

# plt.figure(figsize=(5,25))
# df[df["ori_meta.genre_1"] != ""]["ori_meta.genre_1"].value_counts().plot.barh()
# plt.show()

# plt.figure(figsize=(5,25))
# df[df["ori_meta.genre_1"] != ""]["ori_meta.genre_1"].value_counts().plot.barh()
# plt.show()

# top_genres = df[df["ori_meta.genre_1"] != ""]["ori_meta.genre_1"].value_counts()[:20].keys()
# top_genres = sorted(set(top_genres))
# print(top_genres)

Total duration 1939.744431064815 hours


In [62]:
from recipes.datasets.mir.taxonomies.music_sft_zh import MusicSFTTokenizerGenresZH


vocab_raw = df["playlist_extra.label1"].value_counts().keys().tolist()


tokenizer = MusicSFTTokenizerGenresZH()

In [7]:
from recipes.datasets.mir.taxonomies.music_sft_en import MusicSFTTokenizerGenresEN

tokenizer = MusicSFTTokenizerGenresEN()
print(tokenizer.vocab)

tokenizer.encode("Classical Music", "cpu")

['blues', "children's music", 'classical music', 'country', 'devotional', 'electronic music', 'folk', 'hip hop', 'jazz', 'metal', 'pop', 'r&b', 'reggae', 'rock', 'sound track']


tensor(2)

In [8]:
from recipes.datasets.mir.music_sft import MusicSFTDataModule

datamodule = MusicSFTDataModule(
    tokenizer=tokenizer,
    sample_rate=24000,
    duration=30,
    batch_size=64,
    shuffle_buffer_size=0,
    resampled=False,
    shardshuffle=False,
    num_workers=8
)
train_loader = datamodule.train_dataloader()
batch = next(iter(train_loader))

2024-01-11 09:57:39,602 - bytedance.easycycle.bigspeech - INFO - region: Region.I18n
2024-01-11 09:57:39,603 - bytedance.easycycle.bigspeech - INFO - req body: {"id": 112, "trail_id": "4072173"}, headers: {'content-type': 'application/json'}


/opt/tiger/samantha/recipes/datasets/mir/music_sft.py:34: UserWarning: torchaudio._backend.set_audio_backend has been deprecated. With dispatcher enabled, this function is no-op. You can remove the function call.
  torchaudio.set_audio_backend("soundfile")


2024-01-11 09:57:39,836 - bytedance.easycycle.bigspeech - INFO - post https://bigspeech.byteintl.net/platform/api/v3/data/dataset_collection/get_dataset_collection_info, status code: 200, body: {"paths":[{"data":"hdfs://harunava/home/byte_data_seed_us/hdd_va/speech/data_store/BigMusic/music_sft-en-20231221_Smcc_Mvocal_N6.2k/data//*.parquet","index":"hdfs://harunava/home/byte_data_seed_us/hdd_va/speech/data_store/BigMusic/music_sft-en-20231221_Smcc_Mvocal_N6.2k/index_1//*.parquet"}]}, headers: {'Date': 'Thu, 11 Jan 2024 01:57:39 GMT', 'Content-Type': 'application/json; charset=utf-8', 'Transfer-Encoding': 'chunked', 'Connection': 'keep-alive', 'Vary': 'Accept-Encoding', 'X-Tt-Logid': '2024011101573819B870B511904C0F3465', 'server-timing': 'inner; dur=215', 'x-tt-trace-host': '015b3f4f93d54ed6e51e292f03434c52f5af228b7eb06dbc0b4ca2676fd4fa19a3bc50871d7790b8bdc8512a53b809c42d8f327de325eff54850c221280f2ab9d6876f9ab70d87b2b5a449056debdb89078eeddaf1ef75bf690522052b516b503c6962571239594ec17fbf2

parse_urls:   0%|          | 0/1 [00:00<?, ?it/s]

2024-01-11 09:57:39,838 - samantha.utils.hdfs_helper - INFO - Listing HDFS directory hdfs://harunava/home/byte_data_seed_us/hdd_va/speech/data_store/BigMusic/music_sft-en-20231221_Smcc_Mvocal_N6.2k/data
2024-01-11 09:57:39,934 - samantha.utils.hdfs_helper - INFO - Listing HDFS directory hdfs://harunava/home/byte_data_seed_us/hdd_va/speech/data_store/BigMusic/music_sft-en-20231221_Smcc_Mvocal_N6.2k/index_1


...

2024-01-11 09:57:40,045 - bytedance.easycycle.bigspeech - INFO - region: Region.I18n
2024-01-11 09:57:40,045 - bytedance.easycycle.bigspeech - INFO - req body: {"id": 112, "trail_id": "4072173"}, headers: {'content-type': 'application/json'}
2024-01-11 09:57:40,068 - bytedance.easycycle.bigspeech - INFO - post https://bigspeech.byteintl.net/platform/api/v3/data/dataset_collection/get_dataset_collection_info, status code: 200, body: {"paths":[{"data":"hdfs://harunava/home/byte_data_seed_us/hdd_va/speech/data_store/BigMusic/music_sft-en-20231221_Smcc_Mvocal_N6.2k/data//*.parquet","index":"hdfs://harunava/home/byte_data_seed_us/hdd_va/speech/data_store/BigMusic/music_sft-en-20231221_Smcc_Mvocal_N6.2k/index_1//*.parquet"}]}, headers: {'Date': 'Thu, 11 Jan 2024 01:57:40 GMT', 'Content-Type': 'application/json; charset=utf-8', 'Transfer-Encoding': 'chunked', 'Connection': 'keep-alive', 'Vary': 'Accept-Encoding', 'X-Tt-Logid': '2024011101573994C8AD954AF4750CA7A0', 'server-timing': 'inner; dur


parse_urls:   0%|          | 0/1 [00:00<?, ?it/s]

2024-01-11 09:57:40,070 - samantha.utils.hdfs_helper - INFO - Listing HDFS directory hdfs://harunava/home/byte_data_seed_us/hdd_va/speech/data_store/BigMusic/music_sft-en-20231221_Smcc_Mvocal_N6.2k/data
2024-01-11 09:57:40,234 - samantha.utils.hdfs_helper - INFO - Listing HDFS directory hdfs://harunava/home/byte_data_seed_us/hdd_va/speech/data_store/BigMusic/music_sft-en-20231221_Smcc_Mvocal_N6.2k/index_1


parse_urls: 100%|██████████| 1/1 [00:00<00:00,  2.82it/s]
ERROR: Unexpected segmentation fault encountered in worker.
 ERROR: Unexpected segmentation fault encountered in worker.
 ERROR: Unexpected segmentation fault encountered in worker.
 ERROR: Unexpected segmentation fault encountered in worker.
 ERROR: Unexpected segmentation fault encountered in worker.
 ERROR: Unexpected segmentation fault encountered in worker.
 ERROR: Unexpected segmentation fault encountered in worker.
 ERROR: Unexpected segmentation fault encountered in worker.
 

RuntimeError: DataLoader worker (pid(s) 567640, 567829) exited unexpectedly

In [9]:
!pip3 install -q ujson
import pandas as pd
from recipes.mi1.data_enriched.reader import IndexReader

indexes = []
for tag_name in tokenizer.vocab:
    indexes.append(IndexReader(f"/mnt/bn/janne-research-xl/data/music_sft/english/primary_genre/{tag_name}/url2index.txt").indexes)
    
indexes = pd.concat(indexes)

DEPRECATION: omegaconf 2.0.6 has a non-standard dependency specifier PyYAML>=5.1.*. pip 24.0 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of omegaconf or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063
Loaded 5 index urls


100%|██████████| 5/5 [00:00<00:00, 193.35it/s]


Loaded 2 index urls


100%|██████████| 2/2 [00:00<00:00, 267.42it/s]


Loaded 2 index urls


...

100%|██████████| 3/3 [00:00<00:00, 153.91it/s]


Loaded 11 index urls


100%|██████████| 11/11 [00:00<00:00, 131.60it/s]


Loaded 1 index urls


100%|██████████| 1/1 [00:00<00:00, 174.84it/s]


In [10]:
tag_id = indexes["tag_ids"].values[0]
print(tag_id, tokenizer(indexes["tag_names"].values[0], "cpu"))

tensor(0) tensor(0)


In [11]:
indexes["tag_names"].value_counts()

pop                 18064
rock                 5299
electronic music     3466
jazz                 3419
hip hop              3287
metal                2680
blues                2161
r&b                  2027
country              1772
reggae               1467
folk                 1286
devotional            841
classical music       693
children's music      546
sound track           416
Name: tag_names, dtype: int64

In [ ]:

from recipes.datasets.mir.taxonomies.music_sft_en import MusicSFTTokenizerVoiceGenderEN

voice_tokenizer = MusicSFTTokenizerVoiceGenderEN()
voice_tokenizer.encode("Female,Male", "cpu")

In [26]:
indexes = []
for tag_name in voice_tokenizer.vocab:
    try:
        indexes.append(IndexReader(f"/mnt/bn/janne-research-xl/data/music_sft/english/vocal_gender/{tag_name}/url2index.txt").indexes)
    except:
        pass
indexes = pd.concat(indexes)

Loaded 2 index urls


100%|██████████| 2/2 [00:00<00:00, 220.65it/s]


Loaded 29 index urls


100%|██████████| 29/29 [00:00<00:00, 170.46it/s]


Loaded 1 index urls


...

100%|██████████| 4/4 [00:00<00:00, 161.37it/s]


Loaded 56 index urls


100%|██████████| 56/56 [00:00<00:00, 77.77it/s] 


Loaded 1 index urls


100%|██████████| 1/1 [00:00<00:00, 168.05it/s]


In [27]:
indexes["tag_names"].value_counts()

male             28140
female           14561
female,male       1942
chorus             658
male,chorus        463
female,chorus      316
Name: tag_names, dtype: int64

In [60]:
from samantha.dataio.webdataset.extension import IndexedWebDataset

from recipes.datasets.mir.music_sft import MusicSFTPreprocessedDataModule

datamodule = MusicSFTPreprocessedDataModule(
    tokenizer=tokenizer,
    sample_rate=24000,
    duration=30,
    batch_size=64,
    shuffle_buffer_size=0,
    resampled=False,
    shardshuffle=False,
    num_workers=8
)
train_loader = datamodule.train_dataloader()

batch = next(iter(train_loader))

batch.tag_names

/opt/tiger/samantha/recipes/datasets/mir/music_sft.py:34: UserWarning: torchaudio._backend.set_audio_backend has been deprecated. With dispatcher enabled, this function is no-op. You can remove the function call.
  


initializing datasets...
adding dataset: /mnt/bn/janne-research-xl/data/music_sft/english/primary_genre/blues/url2index.txt
adding dataset: /mnt/bn/janne-research-xl/data/music_sft/english/primary_genre/children's music/url2index.txt
adding dataset: /mnt/bn/janne-research-xl/data/music_sft/english/primary_genre/classical music/url2index.txt
adding dataset: /mnt/bn/janne-research-xl/data/music_sft/english/primary_genre/country/url2index.txt
adding dataset: /mnt/bn/janne-research-xl/data/music_sft/english/primary_genre/devotional/url2index.txt
adding dataset: /mnt/bn/janne-research-xl/data/music_sft/english/primary_genre/electronic music/url2index.txt
adding dataset: /mnt/bn/janne-research-xl/data/music_sft/english/primary_genre/folk/url2index.txt
adding dataset: /mnt/bn/janne-research-xl/data/music_sft/english/primary_genre/hip hop/url2index.txt
adding dataset: /mnt/bn/janne-research-xl/data/music_sft/english/primary_genre/jazz/url2index.txt
adding dataset: /mnt/bn/janne-research-xl/dat

['sound track',
 'pop',
 'folk',
 'rock',
 'metal',
 'r&b',
 'classical music',
 'electronic music',
 'jazz',
 'devotional',
 'folk',
 'classical music',
 'electronic music',
 'metal',
 'hip hop',
 'folk',
 'folk',
 'country',
 'metal',
 'r&b',
 'country',
 'pop',
 "children's music",
 'hip hop',
 'country',
 'jazz',
 'jazz',
 'pop',
 'r&b',
 'country',
 'reggae',
 'metal',
 'classical music',
 'classical music',
 'hip hop',
 'rock',
 'jazz',
 'hip hop',
 'r&b',
 'electronic music',
 'devotional',
 'blues',
 'r&b',
 'electronic music',
 'jazz',
 'folk',
 'reggae',
 'blues',
 'sound track',
 'rock',
 'r&b',
 'country',
 'classical music',
 'sound track',
 'r&b',
 'electronic music',
 'jazz',
 'r&b',
 'blues',
 'jazz',
 'jazz',
 'blues',
 'classical music',
 'sound track']

In [51]:
from recipes.datasets.mir.taxonomies.music_sft_en import get_vocab

genre_1 = get_vocab(df["ori_meta.genre_1"].unique())
genre_2 = get_vocab(df["ori_meta.genre_2"].unique())
mood =  get_vocab(df["ori_meta.mood"].unique())
scene = get_vocab(df["ori_meta.scene"].unique())
voice_char = get_vocab(df["ori_meta.voice_char"].unique())
instrument = get_vocab(df["ori_meta.instrument"].unique())
voice_gender = get_vocab(df["ori_meta.voice_gender"].unique())

2024-01-11 06:06:10,593 - databus.databus_cache - INFO - databus python cache flush thread begin


/opt/tiger/samantha/samantha/dataio/parquet/writer.py:17: UserWarning: simplejson not installed, potential NaN issue in IndexShardWriter.
  warnings.warn("simplejson not installed, potential NaN issue in IndexShardWriter.")


No module named 'flash_attn'


In [53]:
genre_1

['acapella',
 'alternative/indie',
 'bgm',
 'blues',
 'childhood',
 "children's music",
 'chinese style',
 'chinese tradition',
 'classical',
 'classical music',
 'country',
 'devotional',
 'easy listening',
 'edm',
 'electronic',
 'electronic music',
 'epic',
 'experimental',
 'folk',
 'hip hop',
 'jazz',
 'latin',
 'metal',
 'new age',
 'other',
 'pop',
 'punk',
 'r&b',
 'r&b/soul',
 'reggae',
 'rock',
 'sound effect',
 'sound track',
 'tuhai',
 'world music']

# MusicSFT (ZH)

In [10]:
index_files = hdfs_ls("hdfs://haruna/home/byte_data_seed/lf_lq/speech/data_store/BigMusic/music_sft-cn-20231220_Smcc_Mvocal/index_2/*.parquet")
df = pd.concat([pd.read_parquet(f) for f in tqdm(index_files)])
df = pd.io.json.json_normalize(df["meta"].apply(json.loads))


2024-01-09 07:01:20,101 - samantha.utils.hdfs_helper - INFO - Listing HDFS directory hdfs://haruna/home/byte_data_seed/lf_lq/speech/data_store/BigMusic/music_sft-cn-20231220_Smcc_Mvocal/index_2/*.parquet


100%|██████████| 17/17 [00:01<00:00,  9.02it/s]
/tmp/ipykernel_111765/3902880284.py:3: FutureWarning: pandas.io.json.json_normalize is deprecated, use pandas.json_normalize instead.
  df = pd.io.json.json_normalize(df["meta"].apply(json.loads))


In [17]:
genre_1 = get_vocab(df["genre_1"].unique())
genre_2 = get_vocab(df["genre_2"].unique())
mood =  get_vocab(df["mood"].unique())
scene = get_vocab(df["scene"].unique())
theme = get_vocab(df["theme"].unique())
voice_char = get_vocab(df["voice_char"].unique())
instrument = get_vocab(df["instrument"].unique())
voice_gender = get_vocab(df["voice_gender"].unique())

In [18]:
voice_gender

['child', 'chorus', 'female', 'male', 'neutral']

In [1]:
df["voice_gender"]

NameError: name 'df' is not defined